# AgriVision — YOLOv8 Banana Disease Training

Fine-tune **YOLOv8n** on `datasets/yolo_banana` and deploy weights to `models/best.pt` for live inference.

**Before this notebook:** export Label Studio annotations (see `docs/MODEL_TRAINING.md`).

Run cells from top to bottom. Recommended: GPU with CUDA.

In [ ]:
from pathlib import Path
import shutil
import random

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display
from ultralytics import YOLO

# Project root (repo root)
ROOT = Path.cwd()
if not (ROOT / "train.py").is_file():
    ROOT = Path.cwd().parent
if not (ROOT / "train.py").is_file():
    raise FileNotFoundError("Open this notebook from the AgriVision repo (notebooks/ or repo root).")

YOLO_ROOT = ROOT / "datasets" / "yolo_banana"
DATA_YAML = YOLO_ROOT / "data.yaml"
BASE_WEIGHTS = ROOT / "yolov8n.pt"
MODELS_DIR = ROOT / "models"
PROJECT = "runs"
RUN_NAME = "banana_disease"

print("ROOT:", ROOT)
print("Dataset:", DATA_YAML)

## 1. Environment check

In [ ]:
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = "0" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
    print("PyTorch not found — training will use CPU via Ultralytics")

import ultralytics
print("Ultralytics:", ultralytics.__version__)

## 2. Dataset check

Verifies `data.yaml`, image/label counts, and shows one training sample with boxes.

In [ ]:
import yaml
import cv2

if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_YAML}\n"
        "Export Label Studio first:\n"
        "  python tools/label_studio/export_yolo.py --json export.json "
        "--local-files-root C:/path/to/images --output datasets/yolo_banana"
    )

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
class_names = cfg.get("names", {})
if isinstance(class_names, dict):
    id_to_name = {int(k): v for k, v in class_names.items()}
else:
    id_to_name = {i: n for i, n in enumerate(class_names)}

for split in ("train", "val"):
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split
    n_img = len(list(img_dir.glob("*"))) if img_dir.is_dir() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.is_dir() else 0
    print(f"{split}: {n_img} images, {n_lbl} label files")

print("Classes:", id_to_name)

In [ ]:
# Visualize one random training image + labels
train_imgs = sorted((YOLO_ROOT / "images" / "train").glob("*.*"))
if train_imgs:
    sample = random.choice(train_imgs)
    lbl_path = YOLO_ROOT / "labels" / "train" / f"{sample.stem}.txt"
    img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img)
    if lbl_path.is_file():
        for line in lbl_path.read_text(encoding="utf-8").strip().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cid, cx, cy, bw, bh = parts
            cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
            x = (cx - bw / 2) * w
            y = (cy - bh / 2) * h
            rect = patches.Rectangle((x, y), bw * w, bh * h, linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x, y - 4, id_to_name.get(int(cid), cid), color="lime", fontsize=9)
    ax.set_title(sample.name)
    ax.axis("off")
    plt.show()
else:
    print("No training images found.")

## 3. Train

Adjust hyperparameters below. Set `RESUME` to a `last.pt` path to continue an interrupted run.

In [ ]:
EPOCHS = 80
IMGSZ = 640
BATCH = 16
WORKERS = 0  # keep 0 on Windows
RESUME = None  # e.g. ROOT / "runs/detect/runs/banana_disease/weights/last.pt"

weights = str(RESUME) if RESUME else str(BASE_WEIGHTS)
model = YOLO(weights)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=RESUME is None,
    resume=bool(RESUME),
    workers=WORKERS,
    verbose=True,
)

save_dir = Path(results.save_dir)
best_weights = save_dir / "weights" / "best.pt"
print("Run folder:", save_dir)
print("Best weights:", best_weights)

## 4. Keras-style training curves

Generates **model accuracy** and **model loss** plots (train vs valid) like a Keras `History` chart.

- **valid accuracy** → mAP@0.5 from validation
- **train accuracy** → rises as training loss decreases
- **loss** → sum of box + class + DFL losses

Skip training and set `save_dir` manually to plot an existing run.

In [ ]:
import pandas as pd
import numpy as np


def load_run_history(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df


def history_series(df: pd.DataFrame) -> dict[str, np.ndarray]:
    epochs = df["epoch"].to_numpy()
    train_loss = (
        df["train/box_loss"] + df["train/cls_loss"] + df["train/dfl_loss"]
    ).to_numpy()
    val_loss = (
        df["val/box_loss"] + df["val/cls_loss"] + df["val/dfl_loss"]
    ).to_numpy()
    val_acc = df["metrics/mAP50(B)"].to_numpy()
    t0 = float(train_loss[0]) if len(train_loss) else 1.0
    train_acc = np.clip(1.0 - train_loss / max(t0, 1e-9), 0.0, 1.0)
    return {
        "epoch": epochs,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "train_loss": train_loss,
        "val_loss": val_loss,
    }


def plot_keras_style_history(
    csv_path: Path,
    caption: str | None = None,
    save_path: Path | None = None,
) -> plt.Figure:
    """Side-by-side model accuracy + model loss (Keras History style)."""
    series = history_series(load_run_history(csv_path))
    epochs = series["epoch"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, series["train_acc"], label="train", color="C0")
    axes[0].plot(epochs, series["val_acc"], label="valid", color="C1")
    axes[0].set_title("model accuracy")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("accuracy")
    axes[0].legend(loc="lower right")

    axes[1].plot(epochs, series["train_loss"], label="train", color="C0")
    axes[1].plot(epochs, series["val_loss"], label="valid", color="C1")
    axes[1].set_title("model loss")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("loss")
    axes[1].legend(loc="upper right")

    if caption:
        fig.text(0.5, -0.02, caption, ha="center", va="top", fontsize=11)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved -> {save_path}")
    plt.show()
    return fig


def plot_keras_style_compare(
    runs: list[tuple[Path, str]],
    save_path: Path | None = None,
) -> plt.Figure:
    """Stack multiple runs like thesis figure (a) 20 epochs / (b) 40 epochs."""
    n = len(runs)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for row, (run_dir, caption) in enumerate(runs):
        csv_path = run_dir / "results.csv"
        if not csv_path.is_file():
            raise FileNotFoundError(f"Missing {csv_path}")
        series = history_series(load_run_history(csv_path))
        epochs = series["epoch"]
        ax_acc, ax_loss = axes[row]

        ax_acc.plot(epochs, series["train_acc"], label="train", color="C0")
        ax_acc.plot(epochs, series["val_acc"], label="valid", color="C1")
        ax_acc.set_title("model accuracy")
        ax_acc.set_xlabel("epoch")
        ax_acc.set_ylabel("accuracy")
        ax_acc.legend(loc="lower right")

        ax_loss.plot(epochs, series["train_loss"], label="train", color="C0")
        ax_loss.plot(epochs, series["val_loss"], label="valid", color="C1")
        ax_loss.set_title("model loss")
        ax_loss.set_xlabel("epoch")
        ax_loss.set_ylabel("loss")
        ax_loss.legend(loc="upper right")

        ax_loss.text(0.5, -0.28, caption, transform=ax_loss.transAxes, ha="center", fontsize=11)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved -> {save_path}")
    plt.show()
    return fig

In [ ]:
# Plot current training run (or set save_dir to an existing run folder)
if "save_dir" not in globals():
    save_dir = ROOT / "runs" / "detect" / "runs" / RUN_NAME

csv_path = save_dir / "results.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"No results.csv at {csv_path}. Train first or fix save_dir.")

n_epochs = len(load_run_history(csv_path))
caption = f"(a) YOLOv8n Adam Optimizer {n_epochs} epochs"
plot_keras_style_history(
    csv_path,
    caption=caption,
    save_path=save_dir / "model_history_keras_style.png",
)

In [ ]:
# Optional: compare two runs (uncomment and set paths after training 20 vs 40+ epochs)
# plot_keras_style_compare(
#     [
#         (ROOT / "runs/detect/runs/banana_disease_20ep", "(a) YOLOv8n Adam Optimizer 20 epochs"),
#         (ROOT / "runs/detect/runs/banana_disease", "(b) YOLOv8n Adam Optimizer 80 epochs"),
#     ],
#     save_path=ROOT / "output" / "training_compare.png",
# )

### Ultralytics built-in plots (optional)

In [ ]:
for plot_name in ("results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"):
    plot_path = save_dir / plot_name
    if plot_path.is_file():
        display(plot_path.name)
        img = plt.imread(plot_path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

## 5. Validate on val split

In [ ]:
val_model = YOLO(str(best_weights))
metrics = val_model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE)
print(metrics)

## 6. Deploy to AgriVision

Copies `best.pt` to `models/best.pt`. Restart the desktop app to load new weights.

In [ ]:
if not best_weights.is_file():
    raise FileNotFoundError(f"Training weights not found: {best_weights}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
dest = MODELS_DIR / "best.pt"
shutil.copy2(best_weights, dest)
print(f"Deployed -> {dest}")

## 7. Quick inference smoke test

In [ ]:
val_imgs = sorted((YOLO_ROOT / "images" / "val").glob("*.*"))
if val_imgs:
    test_img = val_imgs[0]
    pred = val_model.predict(str(test_img), imgsz=IMGSZ, verbose=False)
    annotated = pred[0].plot()
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Predictions: {test_img.name}")
    plt.axis("off")
    plt.show()
else:
    print("No val images for smoke test.")